In [1]:
import pandas as pd 


In [ ]:
# load parquet file

df_test= pd.read_parquet('../../data/data.parquet')

In [ ]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def tokenise_with_alignment(text, tokenizer):
    words = text.strip().split()

    subword_ids = []
    tokens = []
    word_first_subword = []

    current_index = 0

    for word in words:
        pieces = tokenizer.tokenize(word)

        if not pieces:
            pieces = [tokenizer.unk_token]

        word_first_subword.append(current_index)

        tokens.extend(pieces)
        subword_ids.extend(tokenizer.convert_tokens_to_ids(pieces))

        current_index += len(pieces)

    return words, tokens, subword_ids, word_first_subword


def prepare_dataset(df):
    all_sentences = []
    
    for i in range(len(df)):
        words, tokens, _, word_first_subword = tokenise_with_alignment(
            df["unmasked_text"].iloc[i],
            tokenizer
        )
    
        subword_labels = df["token_entity_labels"].iloc[i]
    
        sentence_words = []
        sentence_labels = []
    
        for w, idx in zip(words, word_first_subword):
            if idx < len(subword_labels):
                label = subword_labels[idx]
            else:
                label = "O"
    
            sentence_words.append(w)
            sentence_labels.append(label)
    
        # keep only valid sentences
        if len(sentence_words) == len(sentence_labels):
            all_sentences.append([sentence_words, sentence_labels])

    return pd.DataFrame(all_sentences, columns=["words", "labels"])


df_test=prepare_dataset(df_test)